# CNN — Convolutional Neural Network

Le `CNN` sert à traiter des **images** (ou signaux structurés en grille). Contrairement au MLP,
ses neurones ne sont pas tous connectés : chaque filtre de convolution ne voit qu'une petite zone locale
(champ récepteur), ce qui exploite la structure spatiale de l'image.

Architecture : une pile de blocs `Conv2d → activation → MaxPool2d`, puis une tête de classification
`AdaptiveAvgPool2d → Flatten → Linear`.

In [12]:
from NeuralNetworks import CNN, Trainer, losses
import torch

## Paramètres du constructeur

| Paramètre | Rôle |
|---|---|
| `in_channels` | canaux de l'image d'entrée (1 = niveaux de gris, 3 = RGB) |
| `output_size` | nombre de classes en sortie |
| `conv_layers` | canaux de sortie de chaque bloc convolutif, ex. `[32, 64, 128]` |
| `kernel_size` | taille du noyau de convolution (3 = 3×3) |
| `norm` | activation après chaque conv (`"Relu"`, `"GELU"`...) |
| `name` | nom du modèle, utilisé pour la sauvegarde |

In [13]:
net = CNN(in_channels=3, output_size=10, conv_layers=[32, 64, 128],
          kernel_size=3, norm="Relu", name="CNN")

## Forward

L'entrée est un tenseur 4D `[batch, in_channels, H, W]`, la sortie est `[batch, output_size]`.
Grâce au `AdaptiveAvgPool2d((1, 1))` final, **n'importe quelle taille d'image H×W est acceptée**.

In [14]:
images = torch.rand(8, 3, 32, 32)    # batch de 8 images 32x32 RGB
print(net(images).shape)             # (8, 10)

images2 = torch.rand(4, 3, 64, 48)   # taille différente : accepté aussi
print(net(images2).shape)            # (4, 10)

torch.Size([8, 10])
torch.Size([4, 10])


## Entraînement

Comme le MLP, le CNN s'entraîne avec le `Trainer` (voir `Trainer.ipynb`).
Pour la classification, on utilise le critère `"CrossEntropy"` avec des labels **one-hot** de shape `[N, output_size]`.

In [15]:
labels = torch.randint(0, 10, (8,))
targets = torch.nn.functional.one_hot(labels, 10).float()   # [8, 10]

In [16]:
trainer = Trainer(net,
                  inputs=images, outputs=targets,
                  init_train_size=1.0, final_train_size=1.0,
                  crit="CrossEntropy", optim="Adam",
                  init_lr=1e-3, final_lr=1e-5,
                  batch_size=4)
trainer.train(500)

Training CNN:   0%|          | 0/500 [00:00<?, ?epoch/s]